In [21]:
!pip install -q google-genai langchain-google-genai langchain-community faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 2.6 MB/s eta 0:00:00


In [22]:
from getpass import getpass
import os

GEMINI_API_KEY = getpass("Enter your Gemini API Key: ")

os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

Enter your Gemini API Key: ··········


In [35]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

vectorstore = FAISS.from_documents(
    documents,
    embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("✅ RAG retriever created successfully!")

✅ RAG retriever created successfully!


In [36]:
vectorstore = FAISS.from_documents(
    documents,
    embeddings
)

print("RAG vector database created successfully!")

RAG vector database created successfully!


In [37]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.4
)

In [38]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.4
)

In [39]:
from google import genai
import os

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

for model in client.models.list():
    if "generateContent" in model.supported_actions:
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.6-preview
models/gemini-robotics-er-2-preview
models/gemini-2.5-computer-use-preview-10-2025
models/an

In [40]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0.4
)

print("Gemini LLM initialized successfully!")

Gemini LLM initialized successfully!


In [41]:
response = llm.invoke(
    "Give me 3 simple travel tips for Goa."
)

print(response.content)

[{'type': 'text', 'text': 'Here are 3 simple travel tips for Goa:\n\n1. **Rent a Scooter to Get Around**\nTaxis in Goa can be quite expensive, and public transport is limited. Renting a scooter (two-wheeler) is the cheapest, most convenient, and most fun way to explore the narrow lanes and beaches. Just make sure to wear a helmet and carry your driving license.\n\n2. **Choose Your Vibe (North vs. South)**\nGoa is divided into two distinct areas. If you want nightlife, water sports, shopping, and crowded beach parties, stay in **North Goa**. If you want peace, clean beaches, nature, and relaxation, head to **South Goa**. \n\n3. **Always Carry Cash**\nWhile digital payments (like UPI and cards) are widely accepted in big restaurants, many beach shacks, local street vendors, parking lots, and water sports operators still prefer cash. Keep some physical currency handy to avoid any hassle.', 'extras': {'signature': 'EoUSCoISARFNMg9Im7kxqvZww6nruwWYjKiwysuZjZ7jSd+TPm8cUkZB5VUeMIQg/R+tmCDr9aj

In [42]:
def create_travel_plan(destination, budget, days, interests):

    # Retrieve relevant travel documents
    query = f"""
    Travel information about {destination}.
    Budget: {budget}.
    Days: {days}.
    Interests: {interests}.
    """

    retrieved_docs = retriever.invoke(query)

    # Combine retrieved documents
    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )

    # Create the prompt
    formatted_prompt = prompt.format(
        destination=destination,
        budget=budget,
        days=days,
        interests=interests,
        context=context
    )

    # Send prompt to Gemini
    response = llm.invoke(formatted_prompt)

    return response.content


print("Travel planner function created successfully!")

Travel planner function created successfully!


In [43]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0.4
)

print("✅ Gemini model ready!")

✅ Gemini model ready!


In [44]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an expert travel planner.

Create a practical travel plan using the user's requirements
and the retrieved travel information.

Destination:
{destination}

Budget:
{budget}

Number of Days:
{days}

Interests:
{interests}

Travel Information:
{context}

Generate the answer using:

1. Trip Overview
2. Day-by-Day Itinerary
3. Hotel Suggestions
4. Packing List
5. Estimated Cost
6. Travel Tips

Keep the plan within the user's budget as much as possible.
Mention that costs are approximate.
Do not claim exact hotel availability or prices.
""")

print("✅ Prompt template ready!")

✅ Prompt template ready!


In [45]:
def create_travel_plan(destination, budget, days, interests):

    query = f"""
    Travel information about {destination}.
    Budget: {budget}.
    Days: {days}.
    Interests: {interests}.
    """

    retrieved_docs = retriever.invoke(query)

    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )

    formatted_prompt = prompt.format(
        destination=destination,
        budget=budget,
        days=days,
        interests=interests,
        context=context
    )

    response = llm.invoke(formatted_prompt)

    return response.content

In [46]:
destination = "Goa"
budget = "₹20,000"
days = 3
interests = "beaches, food and sightseeing"

result = create_travel_plan(
    destination,
    budget,
    days,
    interests
)

print(result)

[{'type': 'text', 'text': 'Here is a practical, budget-friendly 3-day travel plan for your trip to Goa. \n\n---\n\n### 1. Trip Overview\n* **Destination:** Goa (Focusing on North Goa and Panjim for the best mix of sightseeing, beaches, and food within 3 days)\n* **Duration:** 3 Days / 2 Nights\n* **Theme:** Beaches, Portuguese Heritage, Local Seafood, and Sightseeing\n* **Vibe:** Relaxed, cultural, and vibrant\n* **Primary Mode of Transport:** Rental Scooter (highly budget-friendly and convenient)\n\n---\n\n### 2. Day-by-Day Itinerary\n\n#### **Day 1: Beach Vibes & Sunset in North Goa**\n* **Morning:** Arrive in Goa. Check into your hostel or budget hotel in the Anjuna/Calangute area. Rent a scooter right outside the station/airport or near your stay. \n* **Afternoon:** Ride to **Calangute Beach** or **Baga Beach**. Grab a beachside lunch at a local shack. Try classic Goan fish curry rice or Rava fried fish. Relax on the beach.\n* **Evening:** Head to **Anjuna Beach** to catch a gorgeo

In [ ]:
!streamlit run app.py



2026-08-11 05:25:20.894 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.239.52.207:8501

